# Lab 1: Trace an Agent with Python

In this lab, you'll build a simple tracer in Python and use it to record what the Northwind agent does while it answers customer questions. You don't need a special tracing tool: the tracer is a few Python classes, defined in this notebook, that record the agent's main steps and save the results as JSON files.

The agent in this lab lets Claude decide which tools to call. For each question, it sends the question to Claude, runs any tools Claude asks for, sends back the results, and repeats until Claude answers.

The tracer records three kinds of steps, called **spans**:

- a **chain** span for each full question, from the question to the answer
- an **llm** span for each request to Claude
- a **tool** span for each tool the agent runs

These spans don't show everything: a tool span won't show the database queries the tool runs, and an llm span shows what Claude was sent and returned, not how it worked out its answer.

The notebook has six sections:

1. **Setup:** load the code and check your API key.
2. **Build the tracer:** define it in five small parts, then test it on example functions, including one that fails.
3. **Load the shop:** the product catalog and the agent's ten tools.
4. **Connect the tracer to the agent:** its tools, its requests to Claude, and the full question.
5. **Trace one question** and look at the result.
6. **Trace six more questions,** then reload the saved traces and count model calls, tool calls, tokens, time, and errors.

You don't need to write any code. Run each cell in order (`Shift+Enter`) and read the explanation above it. Every question calls Claude, so answers can differ between runs. For background on spans and traces, see **Lab 1 Reading: How Tracing Works**.

## 1: Setup

The next two cells load what the notebook needs:

- **The first** loads Python's standard tools and pandas, points Python to the agent's code in the `apparel-shop-agent` folder, and checks that `ANTHROPIC_API_KEY` is set. On the platform, it already is.
- **The second** loads the agent, the shop's database and tools, and two helper files: `dlai_utils.py`, which cleans and saves trace data, and `trace_display.py`, which draws the trace trees you'll see below.

**Running this on your own computer?** First install the packages by running `pip install -r requirements.txt` from this notebook's folder, and set `ANTHROPIC_API_KEY` to your own key.

In [1]:
import inspect
import os
import sys
import time
import traceback
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field
from functools import wraps
from pathlib import Path
from typing import Any, Callable, Iterator
from uuid import uuid4

import pandas as pd
from IPython.display import display

PACKAGE_ROOT = Path("./apparel-shop-agent").resolve()
SRC_PATH = PACKAGE_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise EnvironmentError("Set ANTHROPIC_API_KEY before running this live-model walkthrough.")

print("Agent package:", PACKAGE_ROOT)
print("ANTHROPIC_API_KEY is set.")


Agent package: /home/jovyan/work/apparel-shop-agent
ANTHROPIC_API_KEY is set.


In [2]:
from apparel_agent.agent.llm_agent import LLMAgent
from apparel_agent.data.database import create_seeded_connection, table_rows
from apparel_agent.data.repository import ApparelRepository
from apparel_agent.tools.tool_registry import ToolSpec, build_tool_registry

from dlai_utils import (
    DEFAULT_REDACTED_KEYS,
    atomic_write_json,
    bind_call_arguments,
    load_trace,
    new_run_id,
    sanitize_value,
    unique_trace_path,
    utc_now,
    walk_spans,
)
from trace_display import show_trace


## 2: Build the tracer

The tracer is built in five parts, each with its own explanation and code cell:

1. **`Span`** (2.1): the record for one operation, with every field it stores.
2. **`SpanRecorder`** (2.2): keeps track of which span is running, so each new span attaches to the right parent, and records when each span starts and ends and whether it failed.
3. **`FunctionTracer`** (2.3): wraps an ordinary function so that calling it creates a span.
4. **`TypedFunctionTracer`** (2.4): adds the `chain`, `llm`, and `tool` decorators.
5. **`LocalTracer`** (2.5): saves a finished trace as a JSON file.

Parts 2 to 5 build on each other: each class adds to the one before it. Sections 2.6 and 2.7 then test the tracer on small example functions, including one that fails.

The helper file `dlai_utils.py` handles supporting details, such as cleaning data before it's saved and writing files safely, so the code here can focus on how tracing works.

### 2.1: The `Span` class

`Span` holds everything recorded about one operation. It's a Python **dataclass**: a class that mainly stores data, with each field and its default value listed at the top.

The fields fall into four groups:

- **What ran:** `name`, and `kind` (`chain`, `llm`, or `tool`).
- **What happened:**
  - `input` and `output`
  - `status`: `unset` until the operation finishes, then `ok` or `error`
  - `error`: the exception's type and message, if it failed
  - `attributes`: any extra details
- **When, and how long:** `start_time` and `end_time` are clock times. `duration_ms` is measured with a separate clock that only moves forward, so it stays accurate even if the computer's clock is adjusted.
- **How it connects:**
  - `trace_id`: shared by every span in one run
  - `span_id`: unique to this span
  - `parent_span_id`: the span this one ran inside
  - `children`: the spans that ran inside this one

`children` and `parent_span_id` record the same link in two ways. `children` keeps the spans as a nested tree, which is easy to read. `parent_span_id` keeps the link even if the spans are later stored as a flat list, which is how many tracing tools store them.

The `to_dict` method turns a span and all its children into plain dictionaries, ready to display or save as JSON. Along the way, it cleans the values with `sanitize_value` from `dlai_utils.py`, which hides API keys, passwords, and email addresses, and shortens very long values.

In [3]:
@dataclass
class Span:
    name: str
    kind: str = "span"  # chain | llm | tool | span
    input: Any = None
    output: Any = None
    attributes: dict[str, Any] = field(default_factory=dict)
    status: str = "unset"
    error: dict[str, str] | None = None
    start_time: str = field(default_factory=utc_now)
    end_time: str | None = None
    duration_ms: float | None = None
    children: list["Span"] = field(default_factory=list)
    trace_id: str = field(default_factory=lambda: uuid4().hex)
    span_id: str = field(default_factory=lambda: uuid4().hex[:16])
    parent_span_id: str | None = None
    _started_at: float = field(default_factory=time.perf_counter, repr=False)

    def to_dict(
        self, sanitizer: Callable[[Any], Any] = sanitize_value
    ) -> dict[str, Any]:
        return {
            "trace_id": self.trace_id,
            "span_id": self.span_id,
            "parent_span_id": self.parent_span_id,
            "name": self.name,
            "kind": self.kind,
            "status": self.status,
            "input": sanitizer(self.input),
            "output": sanitizer(self.output),
            "attributes": sanitizer(self.attributes),
            "error": sanitizer(self.error),
            "start_time": self.start_time,
            "end_time": self.end_time,
            "duration_ms": self.duration_ms,
            "children": [child.to_dict(sanitizer) for child in self.children],
        }


### 2.2 `SpanRecorder`: Keep track of the current span

When one traced function calls another, the new span needs to know its parent: the span that's already running. Passing the current span into every function would mean changing the app's code, so `SpanRecorder` stores it in a **context variable** (Python's `ContextVar`) instead. A context variable works like a global variable, except that each async task gets its own copy, so spans from different tasks don't get mixed up. Each recorder also has its own context variable, so spans from two different tracers never attach to each other.

The recorder's settings control where traces are saved and how values are cleaned: which keys to redact, how long text and lists can be, and whether to save a full traceback when an operation fails.

The main work happens in `start_as_current_span`. It's a **context manager**, used with a `with` block: the span opens when the block starts and closes when it ends. When the block starts, it:

1. creates the span, as the root of a new trace if no span is running, or with the current span's `trace_id` if one is
2. adds it to the list of traces if it's a root, or to its parent's `children` if not
3. makes it the current span, and keeps a token so it can restore the previous span later

When the block ends, it:

4. marks the span `ok`, or, if an exception was raised, marks it `error`, records the exception, and raises it again
5. records the end time and duration, then makes the previous span current again, even if there was an error

Raising the exception again matters. The tracer records the failure without hiding it, so the app behaves exactly as it would without tracing.

In [4]:
class SpanRecorder:
    def __init__(
        self,
        output_dir: str | Path = "traces",
        *,
        redacted_keys: set[str] | frozenset[str] = DEFAULT_REDACTED_KEYS,
        max_string_chars: int = 4_000,
        max_collection_items: int = 100,
        max_depth: int = 12,
        capture_traceback: bool = False,
    ) -> None:
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.redacted_keys = frozenset(key.lower() for key in redacted_keys)
        self.max_string_chars = max_string_chars
        self.max_collection_items = max_collection_items
        self.max_depth = max_depth
        self.capture_traceback = capture_traceback
        self.traces: list[Span] = []
        self.saved_paths: list[Path] = []
        self.run_id = new_run_id()
        self._counter = 0
        self._current_span: ContextVar[Span | None] = ContextVar(
            f"local_current_span_{self.run_id}", default=None
        )

    def sanitize(self, value: Any) -> Any:
        return sanitize_value(
            value,
            redacted_keys=self.redacted_keys,
            max_string_chars=self.max_string_chars,
            max_collection_items=self.max_collection_items,
            max_depth=self.max_depth,
        )

    @contextmanager
    def start_as_current_span(
        self,
        name: str,
        *,
        kind: str = "span",
        input: Any = None,
        attributes: dict[str, Any] | None = None,
    ) -> Iterator[Span]:
        parent = self._current_span.get()
        span = Span(
            name=name,
            kind=kind,
            input=input,
            attributes=attributes or {},
            trace_id=parent.trace_id if parent else uuid4().hex,
            parent_span_id=parent.span_id if parent else None,
        )

        if parent is None:
            self.traces.append(span)
        else:
            parent.children.append(span)

        token = self._current_span.set(span)
        try:
            yield span
        except Exception as exc:
            span.status = "error"
            span.error = {"type": type(exc).__name__, "message": str(exc)}
            if self.capture_traceback:
                span.error["traceback"] = traceback.format_exc()
            raise
        else:
            span.status = "ok"
        finally:
            span.end_time = utc_now()
            span.duration_ms = round(
                (time.perf_counter() - span._started_at) * 1_000, 3
            )
            self._current_span.reset(token)


### 2.3 `FunctionTracer`: Turn functions into spans

A **decorator** is a function that wraps another function to add behavior. You apply it by writing `@` and its name above the function's definition. `FunctionTracer` uses decorators to add tracing: every call to a wrapped function creates a span, without changing what the function does.

`_decorator` takes a span kind (`chain`, `llm`, or `tool`), and optionally a name for the span, and returns a decorator. If you don't give a name, the span uses the function's own name. Each time a wrapped function is called, the wrapper:

1. records the function's arguments, with their parameter names, as the span's input, so an input reads `{"item": "beanie"}` rather than just `("beanie",)`. `bind_call_arguments` from `dlai_utils.py` handles this.
2. opens a span with `start_as_current_span`
3. runs the original function
4. records the result as the span's output
5. returns the result unchanged

`functools.wraps` copies the original function's name and docstring onto the wrapper, and links back to the original function. Tools that look up a function's arguments, such as `help()`, follow that link, so the wrapped function still looks like the original.

Async functions get a separate wrapper that keeps the span open until the function finishes. Calling an async function only sets up its work to run later, so without this wrapper, the span would close before the work even started.

For the agent's chain span, `_span_output` saves a short summary instead of the whole result: the final answer, whether it succeeded, the failure type if there was one, and the names of the tools used. The full tool results are already in the tool spans, so repeating them here would make every trace much larger.

In [5]:
class FunctionTracer(SpanRecorder):
    @staticmethod
    def _span_output(kind: str, result: Any) -> Any:
        if kind == "chain" and hasattr(result, "final_answer"):
            return {
                "final_answer": getattr(result, "final_answer", None),
                "success": getattr(result, "success", None),
                "failure_type": getattr(result, "failure_type", None),
                "tool_names": [
                    getattr(call, "name", None)
                    for call in getattr(result, "tool_calls", []) or []
                ],
            }
        return result

    def _decorator(self, *, kind: str, name: str | None = None):
        def decorate(function: Callable[..., Any]) -> Callable[..., Any]:
            span_name = name or function.__name__

            if inspect.iscoroutinefunction(function):
                @wraps(function)
                async def async_wrapper(*args: Any, **kwargs: Any) -> Any:
                    call_input = bind_call_arguments(function, args, kwargs)
                    with self.start_as_current_span(
                        span_name, kind=kind, input=call_input
                    ) as span:
                        result = await function(*args, **kwargs)
                        span.output = self._span_output(kind, result)
                        return result

                return async_wrapper

            @wraps(function)
            def wrapper(*args: Any, **kwargs: Any) -> Any:
                call_input = bind_call_arguments(function, args, kwargs)
                with self.start_as_current_span(
                    span_name, kind=kind, input=call_input
                ) as span:
                    result = function(*args, **kwargs)
                    span.output = self._span_output(kind, result)
                    return result

            return wrapper

        return decorate


### 2.4 `TypedFunctionTracer`: The `chain`, `llm`, and `tool` decorators

`TypedFunctionTracer` adds three decorators: `chain`, `llm`, and `tool`. All three use the same wrapper from 2.3. The only difference is the span kind they record, which labels the role of each call:

- **`tool`**: a capability the app provides, such as searching the catalog or checking inventory.
- **`llm`**: a request to the model. The span's input holds what was sent, such as the messages, and its output holds the response, including token usage. Like every span, it also records how long the request took.
- **`chain`**: the whole operation, from the question to the answer. The model and tool spans attach beneath it.

You can use each one in two ways:

- **as a decorator**, above a function you're defining: `@tracer.tool`, or `@tracer.tool(name="...")` to give the span a different name
- **by wrapping an existing function**: `tracer.tool(existing_function, name="search_catalog")`. Section 4 uses this to trace the agent's tools and its requests to Claude without editing the agent's code.

In [6]:
class TypedFunctionTracer(FunctionTracer):
    def tool(
        self, function: Callable[..., Any] | None = None, *, name: str | None = None
    ):
        decorate = self._decorator(kind="tool", name=name)
        return decorate(function) if function is not None else decorate

    def llm(
        self, function: Callable[..., Any] | None = None, *, name: str | None = None
    ):
        decorate = self._decorator(kind="llm", name=name)
        return decorate(function) if function is not None else decorate

    def chain(
        self, function: Callable[..., Any] | None = None, *, name: str | None = None
    ):
        decorate = self._decorator(kind="chain", name=name)
        return decorate(function) if function is not None else decorate


### 2.5 `LocalTracer`: Save traces as JSON

`LocalTracer` adds saving. `last_trace` returns the most recent root span, and `save_trace` writes a whole trace, the root and all its children, to one JSON file. `save_last` is a shortcut that saves the most recent trace.

The data is cleaned only when a trace is saved, or shown as a tree. The traced functions still receive and return their original Python objects; only the saved or displayed copy is cleaned. Using the recorder's settings from 2.2, the cleaning:

- replaces values stored under common sensitive keys, such as `api_key`, `password`, `token`, and `email`, with `[REDACTED]`
- masks API keys and bearer tokens that appear inside text
- shortens very long text and very long lists
- stops at a set depth in deeply nested data, and handles data that loops back on itself, so saving never gets stuck

It only recognizes these keys and patterns, so a secret written in ordinary text can still get through.

Each file gets a unique name, built from when the tracer was created plus a random code, a counter, a short version of the trace's input, and part of the root span's ID. For example: `20261028T172718Z_22ef1ce2_0002_product_id_tee_001_e31abe.json`. So restarting the notebook and running it again never overwrites an earlier file.

`atomic_write_json` writes each trace to a temporary file first, then moves it into place, so an interrupted save doesn't leave a half-written file.

In [7]:
class LocalTracer(TypedFunctionTracer):
    def last_trace(self) -> Span | None:
        return self.traces[-1] if self.traces else None

    def save_trace(
        self, span: Span | None = None, path: str | Path | None = None
    ) -> Path:
        root = span or self.last_trace()
        if root is None:
            raise ValueError("No trace available to save.")

        if path is None:
            self._counter += 1
            label = str(root.input) if root.input is not None else root.name
            path = unique_trace_path(
                self.output_dir,
                run_id=self.run_id,
                counter=self._counter,
                label=label,
                span_id=root.span_id,
            )

        payload = root.to_dict(sanitizer=self.sanitize)
        saved_path = atomic_write_json(path, payload)
        self.saved_paths.append(saved_path)
        return saved_path

    def save_last(self, path: str | Path | None = None) -> Path:
        return self.save_trace(self.last_trace(), path=path)


### 2.6 Test the tracer: Nested spans

Before connecting the tracer to the agent, the next cell tests it on three small functions. They're ordinary Python: `lookup_price` and `check_stock` are traced as tools, and `fake_ask` is traced as a chain that calls both.

Because `fake_ask` is the current span while the two tools run, both tool spans become its children. The cell calls `fake_ask("I need something warm")`, saves the trace to the `traces` folder, and shows it as a tree. Check that:

- the two tool spans appear under `fake_ask`
- the printed result is the same as `fake_ask` would return without tracing

This cell also creates `tracer`, the tracer the rest of the notebook uses.

In [8]:
tracer = LocalTracer(output_dir=Path("traces"))

@tracer.tool
def lookup_price(item: str) -> dict:
    prices = {"beanie": 24.0, "classic tee": 18.0}
    return {"item": item, "price": prices.get(item.lower())}

@tracer.tool
def check_stock(item: str) -> dict:
    quantity = {"beanie": 12, "classic tee": 40}.get(item.lower(), 0)
    return {"item": item, "available": quantity > 0, "quantity": quantity}

@tracer.chain
def fake_ask(query: str) -> dict:
    item = "beanie" if "warm" in query.lower() else "classic tee"
    return {"query": query, "price": lookup_price(item), "stock": check_stock(item)}

toy_result = fake_ask("I need something warm")
toy_path = tracer.save_last()
print(toy_result)
print("Saved:", toy_path)
show_trace(tracer.last_trace(), title="Toy chain → tool spans")


{'query': 'I need something warm', 'price': {'item': 'beanie', 'price': 24.0}, 'stock': {'item': 'beanie', 'available': True, 'quantity': 12}}
Saved: traces/20261111T055746Z_de6b4385_0001_query_i_need_something_warm_fa72af.json


### Toy chain → tool spans

- • **fake_ask** `chain` — 0.09 ms
  - input: `{"query": "I need something warm"}`
  - output: `{"query": "I need something warm", "price": {"item": "beanie", "price": 24.0}, "stock": {"item": "beanie", "available": true, "quantity": 12}}`
  - • **lookup_price** `tool` — 0.01 ms
    - input: `{"item": "beanie"}`
    - output: `{"item": "beanie", "price": 24.0}`
  - • **check_stock** `tool` — 0.01 ms
    - input: `{"item": "beanie"}`
    - output: `{"item": "beanie", "available": true, "quantity": 12}`

### 2.7 Test the tracer: A failing function

A failed operation still needs a span, with its input, end time, duration, and error. The next cell traces `unavailable_inventory` as a tool that always raises an error, calls it, catches the error, and shows the saved span.

The recorder finishes the span and makes the previous span current again before the error reaches the caller, so later spans still attach to the right parent. Check that:

- the caller receives the original error, printed as `Caller received: ...`
- the span is marked as failed, with ✗ in the tree, and shows the error's type and message

In [9]:
@tracer.tool
def unavailable_inventory(product_id: str) -> dict:
    raise RuntimeError(f"Inventory service unavailable for {product_id}")

try:
    unavailable_inventory("TEE-001")
except RuntimeError as exc:
    print("Caller received:", exc)

error_path = tracer.save_last()
show_trace(tracer.last_trace(), title="Failed tool span")
print("Saved:", error_path)


Caller received: Inventory service unavailable for TEE-001


### Failed tool span

- ✗ **unavailable_inventory** `tool` — 0.01 ms
  - input: `{"product_id": "TEE-001"}`
  - error: `{"type": "RuntimeError", "message": "Inventory service unavailable for TEE-001"}`

Saved: traces/20261111T055746Z_de6b4385_0002_product_id_tee_001_83ebe0.json


## 3: Load the shop

The agent works from a small shop database stored in SQLite, a database that lives in a single file and needs no server. The next cell creates it as `apparel_shop_lab.sqlite` and fills it with sample data: 20 products, their stock, two orders with tracking details, and some coupons. Because it uses `reset=True`, the database is rebuilt from scratch every time the cell runs, so each run starts from the same data. That matters because some tools change the database, for example `create_cart` and `reserve_item`.

The cell then shows three previews:

- **Products:** the first eight items, with each one's ID, name, category, price, whether it's available, and its care instructions.
- **Inventory:** stock for the first eight combinations of product, size, and store.
- **Tools:** the ten tools the agent can call, with their descriptions. Claude reads these descriptions to decide which tool to use.

You don't need to study the database's layout. What matters for tracing is that the tools read from it, so the tool spans show exactly what data the agent was given. That's what you'll check its answers against.

In [10]:
DB_PATH = Path("apparel_shop_lab.sqlite")
connection = create_seeded_connection(DB_PATH, reset=True)

products = pd.DataFrame(table_rows(connection, "products"))
inventory = pd.DataFrame(table_rows(connection, "inventory"))
display(products[["id", "name", "category", "price", "available", "care"]].head(8))
display(inventory[["product_id", "size", "store_id", "quantity"]].head(8))

repo = ApparelRepository(connection=connection)
tool_registry = build_tool_registry(repo)
display(pd.DataFrame(
    {"tool": spec.name, "description": spec.description}
    for spec in tool_registry.values()
))


,id,name,category,price,available,care
0,TEE-001,Classic Tee,tops,24.99,1,
1,SHO-001,Trail Runner,shoes,89.99,1,
2,SHO-002,Court Shoe,shoes,74.99,1,
3,SHO-003,Slip-On Canvas,shoes,55.00,0,
4,OUT-001,Denim Jacket,outerwear,98.00,1,
5,OUT-002,Raincoat,outerwear,125.00,1,
6,OUT-003,Fleece Pullover,outerwear,68.00,1,
7,TOP-002,Oxford Shirt,tops,68.00,1,


,product_id,size,store_id,quantity
0,TEE-001,XS,STORE-001,42
1,TEE-001,S,STORE-001,42
2,TEE-001,M,STORE-001,42
3,TEE-001,L,STORE-001,42
4,TEE-001,XL,STORE-001,42
5,SHO-001,7,STORE-001,10
6,SHO-001,8,STORE-001,10
7,SHO-001,9,STORE-001,10


,tool,description
0,search_catalog,Search apparel catalog by text query and optio...
1,get_product_details,Get details for one product ID.
2,check_inventory,Check store inventory for a product and size.
3,reserve_item,Reserve one in-stock item for customer pickup.
4,create_cart,Create a cart for a customer with one or more ...
5,apply_coupon,Apply a coupon code to an existing cart.
6,lookup_order,Look up a customer order by order ID and email.
7,get_order_tracking,Get shipment status and tracking events for an...
8,check_return_policy,Check whether an order item is returnable or e...
9,create_exchange_request,Create an exchange request for an eligible ord...


## 4: Connect the tracer to the agent

Now the tracer goes onto the real agent, in three steps, one for each kind of span:

1. **The tools (4.1):** wrap each of the ten tools, so every tool call becomes a `tool` span.
2. **The requests to Claude (4.2):** wrap the agent's model client, so every request becomes an `llm` span.
3. **The full question (4.3):** define `ask_agent`, traced as a `chain`, so every question becomes the root of a trace.

Doing each step separately keeps the tracing visible: you can see exactly which code creates each kind of span. None of the steps edits the agent's own code.

### 4.1 Trace the tools

Each tool is stored as a `ToolSpec`: its name, its description, a description of its inputs, and the function that runs it. A `ToolSpec` can't be changed once it's created, so `instrument_tool` builds a new one for each tool, with the same name, description, and inputs, but with the function wrapped by `tracer.tool`. Each tool span is named after its tool, such as `search_catalog`.

The agent is then given these traced tools in place of the originals. It needs no tracing code of its own: when it calls `search_catalog`, `check_inventory`, or any other tool, the call's inputs and result are recorded as a `tool` span.

In [11]:
def instrument_tool(spec: ToolSpec) -> ToolSpec:
    return ToolSpec(
        name=spec.name,
        description=spec.description,
        input_schema=spec.input_schema,
        func=tracer.tool(spec.func, name=spec.name),
    )

traced_tools = {name: instrument_tool(spec) for name, spec in tool_registry.items()}
print(f"Instrumented {len(traced_tools)} tools.")


Instrumented 10 tools.


### 4.2 Trace the requests to Claude

This cell creates the agent and makes two changes to it.

**It turns off the agent's prerecorded answers.** The agent comes with saved answers for a set of test questions, and when it gets one of those questions, it returns the saved answer instead of asking Claude. Every question in this notebook is on that list, so without this change, none of them would reach Claude and their traces would have no `llm` spans. Clearing the list makes every question a live request to Claude.

**It wraps the agent's model client.** Every request to Claude goes through the client's `create_message` method, so wrapping it with `tracer.llm` turns each request into an `llm` span named `anthropic.messages.create`. Each span records:

- **the request:** the model name, the token limit, the system prompt, the conversation so far, and the tool descriptions
- **the response:** Claude's reply, including any tool requests, the reason it stopped, and how many tokens it used
- **how long the request took**

The span doesn't show how Claude worked out its reply, only what it was sent and what it returned.

The client is wrapped before any question is sent, so every request becomes a span, including the follow-up requests the agent sends after running tools.

In [12]:
agent = LLMAgent(
    traced_tools,
    model_name="claude-haiku-5-5",
    temperature=0.0,
)

agent.fixture_agent._fixtures_by_input.clear()
agent.client.create_message = tracer.llm(
    agent.client.create_message, name="anthropic.messages.create"
)
print("Live model client instrumented.")


Live model client instrumented.


### 4.3 Trace the full question

`ask_agent` is a small function that passes a question to the agent and returns its result. Traced with `@tracer.chain`, it becomes the root span for each question. While it's running, every `llm` and `tool` span attaches beneath it.

So each trace answers one question: what happened, from the moment the customer asked until the agent replied?

The cell also creates `agent_trace_paths`, an empty list that collects the file path of each saved agent trace. Section 6 uses it to reload the traces for analysis.

In [13]:
@tracer.chain
def ask_agent(user_query: str):
    return agent.run(user_query)

agent_trace_paths: list[Path] = []
print("End-to-end chain instrumented.")


End-to-end chain instrumented.


## 5: Trace one question

The next cell asks the agent one question, "What colors does the Classic Tee come in?", saves the trace, and shows it as a tree. It also prints the agent's answer and the tools the agent reports using.

A typical trace has at least two `llm` spans, because the agent works in a loop:

1. Claude asks for a tool, such as `search_catalog`.
2. The agent runs the tool, which appears as a `tool` span.
3. Claude writes the answer from the tool's result.

This is a live request, so the tools Claude chooses and the wording of its answer can change from run to run. When you look at the trace, check that:

- the tool spans in the tree match the tools the agent reports
- the colors in the answer appear in a tool span's output

In [14]:
result = ask_agent("What colors does the Classic Tee come in?")
path = tracer.save_last()
agent_trace_paths.append(path)

print("ANSWER:", result.final_answer)
print("TOOLS REPORTED BY AGENT:", [call.name for call in result.tool_calls])
print("Saved:", path)
show_trace(tracer.last_trace(), title="Live Classic Tee trace")


ANSWER: The Classic Tee (TEE-001) comes in four colors: black, heather gray, navy, and white. It's $24.99 and available in sizes XS, S, M, L, and XL.
TOOLS REPORTED BY AGENT: ['search_catalog']
Saved: traces/20261111T055746Z_de6b4385_0003_user_query_what_colors_does_the_classic__b876f7.json


### Live Classic Tee trace

- • **ask_agent** `chain` — 1686.52 ms
  - input: `{"user_query": "What colors does the Classic Tee come in?"}`
  - output: `{"final_answer": "The Classic Tee (TEE-001) comes in four colors: black, heather gray, navy, and white. It's $24.99 and available in sizes XS, S, M, L, and XL.", "success": true, "failure_type": null, "tool_names": ["search_catalog"]}`
  - • **anthropic.messages.create** `llm` — 912.77 ms
    - input: `{"model": "claude-haiku-5-5", "max_tokens": 800, "system": "You are Northwind Apparel's shopping assistant.\n\nYou can help customers find apparel, check inventory, create carts,\nreserve items, apply coupons, look up order tracking, and...`
    - output: `{"id": "msg_011CgvdRE2EUquUa9TuZAUMk", "container": null, "content": [{"id": "toolu_012EpnB9TCFXfJi71AeeWiMk", "caller": {"type": "direct"}, "input": {"query": "Classic Tee"}, "name": "search_catalog", "type": "tool_use", "toolset_name":...`
  - • **search_catalog** `tool` — 1.25 ms
    - input: `{"query": "Classic Tee"}`
    - output: `{"success": true, "query": "Classic Tee", "filters": {}, "count": 3, "products": [{"id": "TEE-001", "name": "Classic Tee", "category": "tops", "color": "white", "fabric": "combed cotton", "description": "100% combed cotton crew-neck t-sh...`
  - • **anthropic.messages.create** `llm` — 770.70 ms
    - input: `{"model": "claude-haiku-5-5", "max_tokens": 800, "system": "You are Northwind Apparel's shopping assistant.\n\nYou can help customers find apparel, check inventory, create carts,\nreserve items, apply coupons, look up order tracking, and...`
    - output: `{"id": "msg_011CgvdRHScWik12pPBWzzMV", "container": null, "content": [{"citations": null, "text": "The Classic Tee (TEE-001) comes in four colors: black, heather gray, navy, and white. It's $24.99 and available in sizes XS, S, M, L, and ...`

## 6: Trace six more questions

The next cell runs six more questions, each chosen to test something different:

- **A simple lookup:** "How much is the Wool Beanie?"
- **A question with two parts:** the Classic Tee in medium, and the return policy.
- **A vague request:** something comfortable for a long flight.
- **Care advice:** how to care for the merino wool sweater. The catalog only says "Machine washable on wool cycle," so any further advice in the answer didn't come from the data.
- **An order the shop doesn't have:** order #4521, which isn't in the sample data.
- **Several products at once:** a winter outfit with a sweater, pants, and a beanie.

For each question, the cell saves the trace and prints how many model calls and tool calls it took. It then shows a table with one row per question: the total time, the number of model calls, the tools used, the total time spent in tools, the number of errors, the start of the answer, and the trace's file name.

These numbers come from the spans, not from the agent's own summary of its run. So they show things the agent doesn't report, such as how many times it called Claude, any steps that failed, and how long each step took.

In [15]:
test_queries = [
    "How much is the Wool Beanie?",
    "Can I get the Classic Tee in medium, and what's your return policy?",
    "I need something comfortable for a long flight.",
    "How should I care for the merino wool sweater?",
    "Where's my order #4521?",
    "Can you help me put together a winter outfit with a sweater, pants, and a beanie?",
]

batch_rows = []
for query in test_queries:
    result = ask_agent(query)
    root = tracer.last_trace()
    path = tracer.save_last()
    agent_trace_paths.append(path)

    spans = list(walk_spans(root))
    tool_spans = [span for span in spans if span.kind == "tool"]
    llm_spans = [span for span in spans if span.kind == "llm"]
    batch_rows.append({
        "query": query,
        "trace_ms": root.duration_ms,
        "llm_calls": len(llm_spans),
        "tools": ", ".join(span.name for span in tool_spans),
        "tool_ms": round(sum(span.duration_ms or 0 for span in tool_spans), 3),
        "errors": sum(span.status == "error" for span in spans),
        "answer_preview": result.final_answer[:140].replace("\n", " "),
        "trace_file": path.name,
    })
    print(f"{query} → {len(llm_spans)} LLM call(s), {len(tool_spans)} tool call(s)")

batch_summary = pd.DataFrame(batch_rows)
display(batch_summary)


How much is the Wool Beanie? → 2 LLM call(s), 1 tool call(s)
Can I get the Classic Tee in medium, and what's your return policy? → 3 LLM call(s), 2 tool call(s)
I need something comfortable for a long flight. → 2 LLM call(s), 1 tool call(s)
How should I care for the merino wool sweater? → 2 LLM call(s), 1 tool call(s)
Where's my order #4521? → 2 LLM call(s), 1 tool call(s)
Can you help me put together a winter outfit with a sweater, pants, and a beanie? → 2 LLM call(s), 3 tool call(s)


,query,trace_ms,llm_calls,tools,tool_ms,errors,answer_preview,trace_file
0,How much is the Wool Beanie?,2055.588,2,search_catalog,0.507,0,The Wool Beanie (ACC-002) is $22.00. It's a cu...,20261111T055746Z_de6b4385_0004_user_query_how_...
1,"Can I get the Classic Tee in medium, and what'...",3921.555,3,"search_catalog, check_inventory",0.677,0,The Classic Tee (TEE-001) is available in medi...,20261111T055746Z_de6b4385_0005_user_query_can_...
2,I need something comfortable for a long flight.,3423.873,2,search_catalog,0.747,0,I found three options that could work for a lo...,20261111T055746Z_de6b4385_0006_user_query_i_ne...
3,How should I care for the merino wool sweater?,2314.142,2,search_catalog,0.502,0,The merino sweater in our catalog is the **Cre...,20261111T055746Z_de6b4385_0007_user_query_how_...
4,Where's my order #4521?,2642.972,2,get_order_tracking,0.138,0,I couldn't find an order with the number 4521....,20261111T055746Z_de6b4385_0008_user_query_wher...
5,Can you help me put together a winter outfit w...,3557.005,2,"search_catalog, search_catalog, search_catalog",1.348,0,I found one option in each category: - **Crew...,20261111T055746Z_de6b4385_0009_user_query_can_...


### 6.1 Analyze the saved traces

The last cell reloads every saved agent trace from its JSON file, the Classic Tee question from section 5 plus the six above, and analyzes them with ordinary pandas code. Working from the files, rather than the spans still in memory, shows that the analysis only needs the saved traces: you could run it later, or on traces someone else saved.

It shows two tables and a tree:

- **One row per trace:** the total time, the number of model calls and tool calls, the input and output tokens, and the number of errors. The token counts come from the `usage` field in each of Claude's responses, when the response includes it.
- **One row per tool:** how many times it was called, its average and longest time, and how many calls failed. Tools with errors are listed first, then the slowest.
- **The last trace,** reloaded from its file and shown as a tree.

In [16]:
trace_rows = []
tool_rows = []
for path in agent_trace_paths:
    trace = load_trace(path)
    spans = list(walk_spans(trace))
    llm_spans = [span for span in spans if span.get("kind") == "llm"]
    tool_spans = [span for span in spans if span.get("kind") == "tool"]

    input_tokens = 0
    output_tokens = 0
    for span in llm_spans:
        usage = (span.get("output") or {}).get("usage") or {}
        input_tokens += usage.get("input_tokens", 0) or 0
        output_tokens += usage.get("output_tokens", 0) or 0

    query = (trace.get("input") or {}).get("user_query", "")
    trace_rows.append({
        "query": query,
        "duration_ms": trace.get("duration_ms"),
        "llm_calls": len(llm_spans),
        "tool_calls": len(tool_spans),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "errors": sum(span.get("status") == "error" for span in spans),
    })
    for span in tool_spans:
        tool_rows.append({
            "tool": span.get("name"),
            "duration_ms": span.get("duration_ms") or 0,
            "error": int(span.get("status") == "error"),
        })

display(pd.DataFrame(trace_rows))
if tool_rows:
    tool_summary = (
        pd.DataFrame(tool_rows)
        .groupby("tool", as_index=False)
        .agg(
            calls=("tool", "size"),
            avg_ms=("duration_ms", "mean"),
            max_ms=("duration_ms", "max"),
            errors=("error", "sum"),
        )
        .sort_values(["errors", "avg_ms"], ascending=False)
    )
    display(tool_summary)

show_trace(load_trace(agent_trace_paths[-1]), title=f"Reloaded {agent_trace_paths[-1].name}")


,query,duration_ms,llm_calls,tool_calls,input_tokens,output_tokens,errors
0,What colors does the Classic Tee come in?,1686.515,2,1,3704,116,0
1,How much is the Wool Beanie?,2055.588,2,1,3565,117,0
2,"Can I get the Classic Tee in medium, and what'...",3921.555,3,2,6205,439,0
3,I need something comfortable for a long flight.,3423.873,2,1,3767,415,0
4,How should I care for the merino wool sweater?,2314.142,2,1,3735,250,0
5,Where's my order #4521?,2642.972,2,1,3330,244,0
6,Can you help me put together a winter outfit w...,3557.005,2,3,4031,545,0


,tool,calls,avg_ms,max_ms,errors
2,search_catalog,8,0.608,1.245,0
0,check_inventory,1,0.162,0.162,0
1,get_order_tracking,1,0.138,0.138,0


### Reloaded 20261111T055746Z_de6b4385_0009_user_query_can_you_help_me_put_together__254fc8.json

- • **ask_agent** `chain` — 3557.01 ms
  - input: `{"user_query": "Can you help me put together a winter outfit with a sweater, pants, and a beanie?"}`
  - output: `{"final_answer": "I found one option in each category:\n\n- **Crew Neck Sweater** (TOP-004): $72, merino wool. Colors: burgundy, charcoal, cream. Sizes: S, M, L, XL.\n- **Chino Pants** (BOT-001): $58, stretch cotton twill. Colors: khaki,...`
  - • **anthropic.messages.create** `llm` — 1319.67 ms
    - input: `{"model": "claude-haiku-5-5", "max_tokens": 800, "system": "You are Northwind Apparel's shopping assistant.\n\nYou can help customers find apparel, check inventory, create carts,\nreserve items, apply coupons, look up order tracking, and...`
    - output: `{"id": "msg_011CgvdT79YzQ2xQrTPW9zFV", "container": null, "content": [{"signature": "CAQSxwUKEAgTGAI4AUIIdGhpbmtpbmcSDDY77JoJTmWVM3VTFxoM0RRaTEHyP0Z4rCJ2IjAJJz1Yh2elPHUW6Z0F+j9d44oC+EueWrH7AonUvEMJkudna4NTWWek8YXTgWFjSb8q5AQsA6AAR6PCegNJ...`
  - • **search_catalog** `tool` — 0.54 ms
    - input: `{"query": "winter sweater"}`
    - output: `{"success": true, "query": "winter sweater", "filters": {}, "count": 1, "products": [{"id": "TOP-004", "name": "Crew Neck Sweater", "category": "tops", "color": "cream", "fabric": "merino wool", "description": "Merino wool crew-neck swea...`
  - • **search_catalog** `tool` — 0.41 ms
    - input: `{"query": "winter pants"}`
    - output: `{"success": true, "query": "winter pants", "filters": {}, "count": 1, "products": [{"id": "BOT-001", "name": "Chino Pants", "category": "bottoms", "color": "khaki", "fabric": "stretch cotton twill", "description": "Straight-fit cotton ch...`
  - • **search_catalog** `tool` — 0.40 ms
    - input: `{"query": "beanie"}`
    - output: `{"success": true, "query": "beanie", "filters": {}, "count": 1, "products": [{"id": "ACC-002", "name": "Wool Beanie", "category": "accessories", "color": "black", "fabric": "merino wool", "description": "Cuffed merino-wool beanie. One si...`
  - • **anthropic.messages.create** `llm` — 2235.49 ms
    - input: `{"model": "claude-haiku-5-5", "max_tokens": 800, "system": "You are Northwind Apparel's shopping assistant.\n\nYou can help customers find apparel, check inventory, create carts,\nreserve items, apply coupons, look up order tracking, and...`
    - output: `{"id": "msg_011CgvdTCupFGSy13i7jqVuv", "container": null, "content": [{"signature": "CAQSiAcKEAgTGAI4AUIIdGhpbmtpbmcSDHwh0ssTLaNgiVv4lhoMJjoV93BQf9t3ErGAIjDZv4S/zeZT0mnuI4tU1juBsUKNg4wpReW/0YP+temzHH3B3rutp/hc70y069GD1ZIqpQbFcGyR3oqCe1Wh...`

## What you built

You've built a working tracer in ordinary Python. It:

- gives every span an ID and records its trace and its parent, so spans link into a tree
- records three kinds of steps: the full question (`chain`), each request to Claude (`llm`), and each tool call (`tool`)
- records whether each step succeeded, with the error's details if it failed
- times each step with a clock that only moves forward
- traces async functions as well as ordinary ones
- cleans each trace and saves it as a JSON file
- shows traces as readable trees
- turns saved traces into numbers you can compare across questions

It also has deliberate limits. It runs inside this notebook and saves files on this computer. A tracing system for a real app usually adds more, such as:

- following a request across several programs or services, so their spans join one trace
- a way to search and query traces
- dashboards
- sampling, which keeps only some traces when there are too many to store
- rules for how long traces are kept

In Lab 2, you'll trace an agent with Phoenix, a tracing tool that provides several of these, including storage, search, and a web interface for browsing traces.